# Thompson Sampling Helper

Optional next-action helper. This is not the primary decision rule.


In [ ]:
import numpy as np
import pandas as pd
from experiment_analysis import load_config, load_results

config = load_config()
df = load_results(config.results_path, config.metric_columns, config.primary_metric)
threshold = df[config.primary_metric].median()
df["success"] = (df[config.primary_metric] >= threshold).astype(int)
df[["tweet_number", "variant", config.primary_metric, "success"]].head()


In [ ]:
counts = (
    df.groupby("variant")["success"]
    .agg(successes="sum", trials="count")
    .astype(int)
)
counts["failures"] = counts["trials"] - counts["successes"]
counts


In [ ]:
rng = np.random.default_rng(config.random_seed)
samples = {}

for variant, row in counts.iterrows():
    samples[variant] = rng.beta(row["successes"] + 1, row["failures"] + 1)

next_variant = max(samples, key=samples.get)
print("Sampled probabilities:", samples)
print("Next tweet variant:", next_variant)
